In [ ]:
import h5py
import os
import subprocess
from anndata.io import read_elem
import cellxgene_mods as cxgm

Define the directory that contains the AnnData file and the unfiltered fragments files

In [ ]:
my_dir = ''

Read in the barcodes from the AnnData & build a barcode lookup for each prefix/suffix\
These patterns will be the values in `id_map` below\
The organism in uns will define the reference chromosme tables

In [ ]:
mx_file = ''

with h5py.File(f'{my_dir}{mx_file}', 'r') as f:
    obs_index = read_elem(f['obs'][f['obs'].attrs['_index']])
    organism = read_elem(f['uns']['organism_ontology_term_id'])

mx_libs = cxgm.build_libraries(obs_index)
set(mx_libs)

Define the ending that is expected on each unfiltered fragments file\
Extract the unique IDs in the names of the raw fragments files\
These will be the keys in `id_map` below

In [ ]:
file_end = '_fragments.tsv.gz'

filtered_end = '_filtered_fragments.tsv.gz'
file_ids = [f[:-len(file_end)] for f in os.listdir(my_dir)
            if f.endswith(file_end) and not f.endswith(filtered_end)]
width = max(map(len, file_ids), default=0)
for file_id in file_ids:
    filt_path = f'{my_dir}{file_id}{filtered_end}'
    note = 'filtered file exists' if os.path.exists(filt_path) else ''
    print(f'{file_id:<{width}}  {note}')

Map the file IDs to the matrix library IDs

In [ ]:
id_map = {
    'G120_F1_N_': 'B@RCODE-1_1',
    'G133_D_FL_': 'B@RCODE-1_4',
    'G210_D_': 'B@RCODE-1_11',
    'G150_D_': 'B@RCODE-1_7',
    'G171_D_': 'B@RCODE-1_9',
    'G120_D_FL_': 'B@RCODE-1_3',
    'G187_D_': 'B@RCODE-1_10',
    'G159_D_': 'B@RCODE-1_8',
    'G129_D_': 'B@RCODE-1_6',
    'G133_N_FL_': 'B@RCODE-1_5',
    'G120_D_TL_': 'B@RCODE-1_2'
}

Review the mapping for any incorrect or unmapped data

In [ ]:
cxgm.check_id_map(id_map, mx_libs, file_ids)

Create a new fragments file for each raw file, amending barcodes to match the matrix barcodes and filtering out non-cells

In [ ]:
results,missing = cxgm.curate_fragments(mx_libs, id_map, my_dir, file_end, filtered_end, organism)
results.to_csv(f'{my_dir}filter_fragments_stats.csv')
results

Review matrix barcodes with no fragments in the associated file

In [ ]:
if missing:
    with open(f'{my_dir}barcodes_without_fragments.txt', 'w') as f:
        f.writelines(f'{m}\n' for m in missing)
    print(f'{len(missing)} barcodes without fragments, written to file')

Concatenate all of the outputs to a single file for validation and submission

In [ ]:
done = results.index[results['', 'error'].isna()]
if len(done):
    concat_frags = f'{my_dir}concatenated_filtered_fragments.tsv.gz'
    with open(concat_frags, 'wb') as out:
        subprocess.run(['cat', *[f'{my_dir}{file_id}{filtered_end}' for file_id in done]], stdout=out, check=True)

**Run the CELLxGENE validator on the revised file**\
This is the same as running `cellxgene-schema process-fragment` in the terminal

In [ ]:
# valid = cxgm.validate(f'{my_dir}{mx_file}', concat_frags)